In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# 1. Laad je samengestelde CSV-bestand
df = pd.read_csv("driver_race_pit_analysis_1994_2022.csv")

# 2. Selecteer relevante kolommen en verwijder ontbrekende waarden
df = df.dropna(subset=["start_pos", "pit_time_pct", "positions_gained"])[
    ["start_pos", "pit_time_pct", "positions_gained"]
]

# 3. Zet startpositie naar gehele getallen
df["start_pos"] = df["start_pos"].astype(int)

# 4. Definieer vaste grid: startposities 1 t/m 24 (X), 20 bins pit_time_pct (Y)
x_vals = np.arange(1, 25)  # Startpositie
y_bins = np.linspace(df["pit_time_pct"].min(), df["pit_time_pct"].max(), 20)
y_vals = (y_bins[:-1] + y_bins[1:]) / 2  # Middens van de pit_time bins

# 5. Bereken Z: gemiddeld aantal gewonnen posities per (start_pos, pit_bin)
Z = np.full((len(x_vals), len(y_vals)), np.nan)

for i, x in enumerate(x_vals):
    subset = df[df["start_pos"] == x]
    inds = np.digitize(subset["pit_time_pct"], y_bins) - 1
    for j in range(len(y_vals)):
        vals = subset.loc[inds == j, "positions_gained"]
        if len(vals) >= 5:
            Z[i, j] = vals.mean()

# 6. Maak 3D surface plot, let op: Z moet getransposed worden!
fig = go.Figure(data=go.Surface(
    x=x_vals,             # Startpositie
    y=y_vals,             # Relatieve pit time
    z=Z.T,                # Transpose nodig voor juiste as-toewijzing
    colorscale=[
    [0.0, "indigo"],    # lage waarden (weinig posities gewonnen)
    [0.5, "violet"],    # middengebied
    [1.0, "red"]        # hoge waarden (veel posities gewonnen)
    ],
    cmin=0,
    cmax=8,
    colorbar=dict(title="Mean Positions\nGained")
))

# 7. Layout instellingen
fig.update_layout(
    title="3D Surface: Startpositie vs. Relatieve Pitstoptijd vs. Posities Gewonnen",
    scene=dict(
        xaxis_title="Startpositie",
        yaxis_title="Pit Time (rel. to fastest = 1.0)",
        zaxis_title="Mean Posities Gained",
        xaxis=dict(dtick=1),
        yaxis=dict(tickformat=".2f")
    ),
    width=780,
    height=700,
    margin=dict(l=65, r=50, b=65, t=90),
    template="plotly_white"
)

fig.show()